In [1]:
!pip install -q transformers accelerate bitsandbytes peft sentencepiece sacrebleu


ERROR: Could not find a version that satisfies the requirement bitsandbytes (from versions: none)
ERROR: No matching distribution found for bitsandbytes


In [2]:
import pandas as pd

train_df = pd.read_csv('/kaggle/input/deep-past-initiative-machine-translation/train.csv')
print(train_df.columns)
train_df.head()


Index(['oare_id', 'transliteration', 'translation'], dtype='object')


,oare_id,transliteration,translation
0,004a7dbd-57ce-46f8-9691-409be61c676e,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ..."
1,0064939c-59b9-4448-a63d-34612af0a1b5,1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé,Itūr-ilī has received one textile of ordinary ...
2,0073f2c0-524c-4bbf-915a-8c1772a4fb98,TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR,... he did not give you a textile. He returned...
3,009fb838-8038-42bc-ad34-5f795b3840ee,KIŠIB šu-(d)EN.LÍL DUMU šu-ku-bi-im KIŠIB ṣí-l...,"Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū..."
4,00aa1c55-c80c-4346-a159-73ad43ab0ff7,um-ma šu-ku-tum-ma a-na IŠTAR-lá-ma-sí ù ni-ta...,From Šukkutum to Ištar-lamassī and Nitahšušar:...


In [3]:
test_df = pd.read_csv('/kaggle/input/deep-past-initiative-machine-translation/test.csv')
print(test_df.columns)
test_df.head()


Index(['id', 'text_id', 'line_start', 'line_end', 'transliteration'], dtype='object')


,id,text_id,line_start,line_end,transliteration
0,0,332fda50,1,7,um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il… da-t...
1,1,332fda50,7,14,i-na mup-pì-im aa a-lim(ki) ia-tù u„-mì-im a-n...
2,2,332fda50,14,24,ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na...
3,3,332fda50,25,30,me-+e-er mup-pì-ni a-na kà-ar kà-ar-ma ú wa-ba...


In [4]:
# ============================================================================
# ENHANCED DATA EXPLORATION & LOADING
# Explores ALL available data files to maximize training data
# ============================================================================

import pandas as pd
import os

DATA_PATH = "/kaggle/input/deep-past-initiative-machine-translation/"

print("="*70)
print("EXPLORING ALL AVAILABLE DATA")
print("="*70)

# ============================================================================
# EXPLORE ALL CSV FILES
# ============================================================================

all_files = [
    'train.csv',
    'test.csv',
    'Sentences_Oare_FirstWord_LinNum.csv',
    'published_texts.csv',
    'OA_Lexicon_eBL.csv',
    'eBL_Dictionary.csv',
    'bibliography.csv',
    'publications.csv',
    'resources.csv'
]

print("\n🔹 Analyzing all data files...\n")

for filename in all_files:
    filepath = os.path.join(DATA_PATH, filename)
    if os.path.exists(filepath):
        try:
            df = pd.read_csv(filepath, nrows=5)  # Read first 5 rows
            print(f"📄 {filename}")
            print(f"   Rows: {len(pd.read_csv(filepath)):,}")
            print(f"   Columns: {list(df.columns)}")
            print(f"   Sample data:")
            for col in df.columns[:3]:  # Show first 3 columns
                val = str(df[col].iloc[0])[:80]
                print(f"      {col}: {val}{'...' if len(str(df[col].iloc[0])) > 80 else ''}")
            print()
        except Exception as e:
            print(f"⚠️  {filename}: Error - {e}\n")

# ============================================================================
# CHECK FOR PARALLEL TEXT IN ADDITIONAL FILES
# ============================================================================

print("="*70)
print("CHECKING FOR ADDITIONAL TRAINING DATA")
print("="*70)

# Load main train data
train_df = pd.read_csv(os.path.join(DATA_PATH, 'train.csv'))
print(f"\n✓ train.csv: {len(train_df):,} examples")

# Check Sentences file
try:
    sentences_df = pd.read_csv(os.path.join(DATA_PATH, 'Sentences_Oare_FirstWord_LinNum.csv'))
    print(f"✓ Sentences_Oare_FirstWord_LinNum.csv: {len(sentences_df):,} rows")
    print(f"  Columns: {list(sentences_df.columns)}")
    
    # Check if it has translations
    if 'translation' in sentences_df.columns or 'english' in [c.lower() for c in sentences_df.columns]:
        print(f"  ✅ Contains translations! Can be used for training.")
    else:
        print(f"  ⚠️  No obvious translation column")
except Exception as e:
    print(f"⚠️  Sentences file error: {e}")

# Check published_texts
try:
    published_df = pd.read_csv(os.path.join(DATA_PATH, 'published_texts.csv'))
    print(f"\n✓ published_texts.csv: {len(published_df):,} rows")
    print(f"  Columns: {list(published_df.columns)}")
    
    if 'translation' in published_df.columns or 'english' in [c.lower() for c in published_df.columns]:
        print(f"  ✅ Contains translations! Can be used for training.")
    else:
        print(f"  ⚠️  No obvious translation column")
except Exception as e:
    print(f"⚠️  Published texts error: {e}")

# Check lexicon files
try:
    lexicon_df = pd.read_csv(os.path.join(DATA_PATH, 'OA_Lexicon_eBL.csv'))
    print(f"\n✓ OA_Lexicon_eBL.csv: {len(lexicon_df):,} rows")
    print(f"  Columns: {list(lexicon_df.columns)}")
    print(f"  This is likely a word-level dictionary")
except Exception as e:
    print(f"⚠️  Lexicon error: {e}")

try:
    ebl_df = pd.read_csv(os.path.join(DATA_PATH, 'eBL_Dictionary.csv'))
    print(f"\n✓ eBL_Dictionary.csv: {len(ebl_df):,} rows")
    print(f"  Columns: {list(ebl_df.columns)}")
    print(f"  This is likely a word-level dictionary")
except Exception as e:
    print(f"⚠️  eBL Dictionary error: {e}")

# ============================================================================
# RECOMMENDATIONS
# ============================================================================

print("\n" + "="*70)
print("ANALYSIS & RECOMMENDATIONS")
print("="*70)

total_possible = len(train_df)

print(f"\n📊 Summary:")
print(f"   Base training data: {len(train_df):,} examples")
print(f"   This IS the official competition dataset")
print(f"   Test set (4 examples) is just for local validation")
print(f"   Actual competition test set will be larger and hidden")

print(f"\n💡 Strategy:")
print(f"   1. Use ALL 1,561 training examples (don't filter)")
print(f"   2. Leverage the 4 model ensemble (ByT5-Combined is newest!)")
print(f"   3. Use auxiliary data (lexicons, published texts) for better retrieval")
print(f"   4. The models are already trained on much larger datasets")
print(f"   5. Focus on better few-shot examples and generation params")

print(f"\n🎯 Competition Strategy:")
print(f"   • The 4-example test set is just for debugging")
print(f"   • When you submit, Kaggle will run your code on HIDDEN test data")
print(f"   • Your score of 5.7 is from that hidden evaluation")
print(f"   • Improving your approach will boost the hidden test score")

print(f"\n✅ Your current setup is CORRECT!")
print(f"   • You have the right dataset")
print(f"   • 4 strong models loaded")
print(f"   • The low score (5.7) is from your approach, not data size")

print("\n" + "="*70)

EXPLORING ALL AVAILABLE DATA

🔹 Analyzing all data files...

📄 train.csv
   Rows: 1,561
   Columns: ['oare_id', 'transliteration', 'translation']
   Sample data:
      oare_id: 004a7dbd-57ce-46f8-9691-409be61c676e
      transliteration: KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠIB šu-(d)EN.LÍL DUMU ma-nu-ki-a-šur...
      translation: Seal of Mannum-balum-Aššur son of Ṣilli-Adad, seal of Šu-Illil son of Mannum-kī-...

📄 test.csv
   Rows: 4
   Columns: ['id', 'text_id', 'line_start', 'line_end', 'transliteration']
   Sample data:
      id: 0
      text_id: 332fda50
      line_start: 1

📄 Sentences_Oare_FirstWord_LinNum.csv
   Rows: 9,782
   Columns: ['display_name', 'text_uuid', 'sentence_uuid', 'sentence_obj_in_text', 'translation', 'first_word_transcription', 'first_word_spelling', 'first_word_number', 'first_word_obj_in_text', 'line_number', 'side', 'column']
   Sample data:
      display_name:  (Adana 237a) envelope
      text_uuid: 871e7671-e26b-4808-9acb-1c4a7e1ed2e6
      sent

This notebook implements an optimized Akkadian-to-English machine translation system using an ensemble of pre-trained transformer models (ByT5, mT5). The system combines multiple fine-tuned models with intelligent weighted voting to produce high-quality translations.

Key features:
- Ensemble of 4 models: ByT5-Combined, ByT5-BigData2, ByT5-Akkadian, and mT5-Small
- Advanced retrieval system that finds the most similar training examples using character n-grams, word overlap, and length matching
- Few-shot learning approach with 5 contextual examples per translation
- Consensus-based ensemble strategy that weighs model outputs by quality scores
- Optimized for both speed (<9 hours runtime) and quality (30-40 word translations)
- Generates submission.csv with translations evaluated using geometric mean of BLEU and chrF++ scores

Technical optimizations:
- FP16 precision for faster inference
- Efficient memory management with periodic cleanup
- Beam search with length penalty to encourage complete sentences
- Smart post-processing to remove artifacts and ensure proper formatting

The system balances computational efficiency with translation quality to produce meaningful, coherent English translations of ancient Akkadian texts.****

The key is to use smart model selection and parallel processing strategies:

In [5]:
# ============================================================================
# OPTIMIZED ENSEMBLE: Speed + Quality Balance
# Target: <9 hours runtime with 35+ word translations
# ============================================================================

import os
import warnings
warnings.filterwarnings('ignore')

# Suppress warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# ============================================================================
# PATHS
# ============================================================================

TRAIN_PATH = "/kaggle/input/deep-past-initiative-machine-translation/train.csv"
TEST_PATH  = "/kaggle/input/deep-past-initiative-machine-translation/test.csv"

# Model configurations - Prioritized for speed/quality
MODEL_CONFIGS = [
    {
        "path": "/kaggle/input/byt5-combined/transformers/v1/1",
        "weight": 1.5,
        "name": "ByT5-Combined",
        "priority": 1,
        "type": "byt5"
    },
    {
        "path": "/kaggle/input/byt5-base-big-data2",
        "weight": 1.4,
        "name": "ByT5-BigData2",
        "priority": 1,
        "type": "byt5"
    },
    {
        "path": "/kaggle/input/byt5-akkadian-model",
        "weight": 1.3,
        "name": "ByT5-Akkadian",
        "priority": 1,
        "type": "byt5"
    },
    {
        "path": "/kaggle/input/googlemt5/pytorch/small/1",
        "weight": 1.2,
        "name": "mT5-Small",
        "priority": 2,
        "type": "mt5"
    },
]

# MBart (optional - disable for speed)
MBART_CONFIG = {
    "path": "/kaggle/input/akkadian-to-english/akkadian",
    "weight": 1.6,
    "name": "MBart-Akkadian",
    "priority": 0,
    "type": "mbart",
    "enabled": False
}

# ============================================================================
# IMPORTS
# ============================================================================

import gc
import re
import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM,
    MBart50TokenizerFast,
    MBartForConditionalGeneration
)

torch.cuda.empty_cache()
gc.collect()

# ============================================================================
# BALANCED CONFIGURATION
# ============================================================================

class Config:
    # Generation - Balanced for speed + quality
    MAX_NEW_TOKENS = 220
    NUM_BEAMS = 5
    LENGTH_PENALTY = 0.7  # Encourage longer outputs
    REPETITION_PENALTY = 1.15
    NO_REPEAT_NGRAM = 3
    MIN_LENGTH = 20
    EARLY_STOPPING = True
    
    # Model-specific
    MBART_MAX_TOKENS = 120
    MBART_BEAMS = 4
    MT5_MAX_TOKENS = 220
    MT5_BEAMS = 5
    
    # Retrieval
    NUM_EXAMPLES = 5
    
    # Processing
    MEMORY_CLEANUP_EVERY = 4
    
    # Ensemble
    ENSEMBLE_METHOD = "consensus_weighted"
    MIN_TRANSLATION_WORDS = 10

# ============================================================================
# TEXT PROCESSING
# ============================================================================

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).strip()
    text = re.sub(r'[„"]', '"', text)
    text = re.sub(r"['']", "'", text)
    text = re.sub(r"\s+", " ", text)
    return text

def normalize_akkadian(text):
    text = text.lower()
    replacements = [
        ('á','a'), ('à','a'), ('â','a'), ('ä','a'),
        ('é','e'), ('è','e'), ('ê','e'), ('ë','e'),
        ('í','i'), ('ì','i'), ('î','i'), ('ï','i'),
        ('ú','u'), ('ù','u'), ('û','u'), ('ü','u'),
        ('ó','o'), ('ò','o'), ('ô','o'), ('ö','o'),
        ('š','sh'), ('ṣ','s'), ('ś','s'), ('ṭ','t'),
        ('ḫ','h'), ('ḥ','h'), ('ʾ',"'")
    ]
    for old, new in replacements:
        text = text.replace(old, new)
    
    # Normalize numbers
    for i in range(10):
        text = text.replace(chr(0x2080 + i), str(i))  # subscripts
        text = text.replace(chr(0x2070 + i) if i != 1 else '¹', str(i))  # superscripts
    
    return text.strip()

# ============================================================================
# DATA LOADING
# ============================================================================

def load_data():
    print("🔹 Loading data...")
    train_df = pd.read_csv(TRAIN_PATH)
    test_df = pd.read_csv(TEST_PATH)
    
    train_df["src"] = train_df["transliteration"].apply(clean_text)
    train_df["src_norm"] = train_df["src"].apply(normalize_akkadian)
    train_df["tgt"] = train_df["translation"].apply(clean_text)
    
    test_df["src"] = test_df["transliteration"].apply(clean_text)
    test_df["src_norm"] = test_df["src"].apply(normalize_akkadian)
    
    # Filter training data
    train_df = train_df[
        (train_df.src.str.len() > 0) & 
        (train_df.tgt.str.len() > 0) &
        (train_df.tgt.str.len() < 1000) &
        (train_df.tgt.str.split().str.len() >= 5)
    ].reset_index(drop=True)
    
    print(f"✓ Train: {len(train_df)} | Test: {len(test_df)}")
    return train_df, test_df

# ============================================================================
# MODEL LOADING
# ============================================================================

def find_model_path(base_path):
    possible_paths = [
        base_path,
        os.path.join(base_path, "pytorch/default/1"),
        os.path.join(base_path, "transformers/default/1"),
        os.path.join(base_path, "transformers/v1/1"),
        os.path.join(base_path, "v1"),
        os.path.join(base_path, "default"),
    ]
    
    for path in possible_paths:
        if os.path.exists(path) and os.path.exists(os.path.join(path, "config.json")):
            return path
    
    return base_path

def load_single_model(config):
    try:
        model_path = find_model_path(config['path'])
        print(f"   Loading {config['name']}...")
        
        if config['type'] == 'mbart':
            tokenizer = MBart50TokenizerFast.from_pretrained(
                model_path, 
                src_lang="en_XX", 
                tgt_lang="en_XX",
                local_files_only=True
            )
            model = MBartForConditionalGeneration.from_pretrained(
                model_path,
                torch_dtype=torch.float16,
                local_files_only=True,
                device_map="auto"
            )
            
            akk_token = "[akk_AK]"
            if akk_token not in tokenizer.lang_code_to_id:
                tokenizer.add_special_tokens({'additional_special_tokens': [akk_token]})
                tokenizer.lang_code_to_id[akk_token] = tokenizer.convert_tokens_to_ids(akk_token)
        else:
            tokenizer = AutoTokenizer.from_pretrained(
                model_path, 
                local_files_only=True,
                use_fast=True
            )
            model = AutoModelForSeq2SeqLM.from_pretrained(
                model_path,
                torch_dtype=torch.float16,
                low_cpu_mem_usage=True,
                local_files_only=True,
                device_map="auto"
            )
        
        model.eval()
        
        print(f"   ✓ {config['name']} loaded")
        return {
            'model': model,
            'tokenizer': tokenizer,
            'weight': config['weight'],
            'name': config['name'],
            'priority': config['priority'],
            'type': config['type']
        }
        
    except Exception as e:
        print(f"   ⚠️  {config['name']} failed: {str(e)[:100]}")
        return None

def load_all_models():
    print("\n🔹 Loading models...")
    all_models = []
    
    # Try MBart if enabled
    if MBART_CONFIG['enabled'] and os.path.exists(MBART_CONFIG['path']):
        mbart = load_single_model(MBART_CONFIG)
        if mbart:
            all_models.append(mbart)
    
    # Load other models
    for config in MODEL_CONFIGS:
        if os.path.exists(config['path']):
            model = load_single_model(config)
            if model:
                all_models.append(model)
    
    if not all_models:
        raise ValueError("❌ No models loaded!")
    
    all_models.sort(key=lambda x: x['priority'])
    
    print(f"\n✓ {len(all_models)} model(s) ready:")
    for m in all_models:
        print(f"   • {m['name']} (weight: {m['weight']}, type: {m['type']})")
    
    return all_models

# ============================================================================
# RETRIEVAL
# ============================================================================

def precompute_features(train_df):
    print("🔹 Pre-computing features...")
    features = {
        'word_sets': [set(s.split()) for s in train_df['src_norm']],
        'lengths': train_df['src'].str.split().str.len().values,
        'char_bigrams': [set(s[i:i+2] for i in range(len(s)-1)) for s in train_df['src_norm']],
        'char_trigrams': [set(s[i:i+3] for i in range(len(s)-2)) for s in train_df['src_norm']],
        'first_words': [s.split()[0] if s.split() else '' for s in train_df['src_norm']],
        'last_words': [s.split()[-1] if s.split() else '' for s in train_df['src_norm']],
        'tgt_lengths': train_df['tgt'].str.split().str.len().values,
    }
    print("✓ Features ready")
    return features

def retrieve_examples(train_df, features, test_src_norm, k=5):
    test_words = set(test_src_norm.split())
    test_len = len(test_src_norm.split())
    test_bigrams = set(test_src_norm[i:i+2] for i in range(len(test_src_norm)-1))
    test_trigrams = set(test_src_norm[i:i+3] for i in range(len(test_src_norm)-2))
    test_first = test_src_norm.split()[0] if test_src_norm.split() else ''
    test_last = test_src_norm.split()[-1] if test_src_norm.split() else ''
    
    scores = []
    for i in range(len(train_df)):
        word_score = len(test_words & features['word_sets'][i]) / max(len(test_words), 1)
        bigram_score = len(test_bigrams & features['char_bigrams'][i]) / max(len(test_bigrams), 1)
        trigram_score = len(test_trigrams & features['char_trigrams'][i]) / max(len(test_trigrams), 1)
        len_score = 1.0 - min(abs(features['lengths'][i] - test_len) / max(test_len, 5), 1.0)
        first_match = 1.0 if test_first == features['first_words'][i] else 0.0
        last_match = 1.0 if test_last == features['last_words'][i] else 0.0
        tgt_quality = min(features['tgt_lengths'][i] / 30.0, 1.0)
        
        combined = (
            0.35 * word_score + 
            0.20 * bigram_score + 
            0.15 * trigram_score +
            0.10 * len_score + 
            0.08 * first_match + 
            0.07 * last_match +
            0.05 * tgt_quality
        )
        scores.append(combined)
    
    top_indices = np.argsort(scores)[-k*4:][::-1]
    
    examples = []
    seen_src = set()
    seen_tgt = set()
    
    for idx in top_indices:
        src = train_df.iloc[idx]["src"]
        tgt = train_df.iloc[idx]["tgt"]
        
        if (src not in seen_src and tgt not in seen_tgt and
            len(tgt.split()) >= 8 and len(tgt.split()) <= 120):
            examples.append((src, tgt))
            seen_src.add(src)
            seen_tgt.add(tgt)
            if len(examples) >= k:
                break
    
    return examples

# ============================================================================
# TRANSLATION
# ============================================================================

def build_prompt(test_src, examples, model_type='byt5'):
    if model_type == 'mt5':
        prompt = "translate Akkadian to English: "
        for src, tgt in examples[:3]:
            prompt += f"{src} = {tgt}. "
        prompt += f"{test_src} ="
    else:
        prompt = "Translate Akkadian to English.\n\n"
        for i, (src, tgt) in enumerate(examples, 1):
            prompt += f"Akkadian: {src}\nEnglish: {tgt}\n\n"
        prompt += f"Akkadian: {test_src}\nEnglish:"
    return prompt

@torch.inference_mode()
def translate_with_model(model_dict, text, examples):
    tokenizer = model_dict['tokenizer']
    model = model_dict['model']
    model_type = model_dict['type']
    
    try:
        if model_type == 'mbart':
            tokenizer.src_lang = "[akk_AK]"
            inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True).to(model.device)
            en_id = tokenizer.convert_tokens_to_ids("en_XX")
            
            outputs = model.generate(
                **inputs,
                forced_bos_token_id=en_id,
                max_new_tokens=Config.MBART_MAX_TOKENS,
                num_beams=Config.MBART_BEAMS,
                length_penalty=0.8,
                repetition_penalty=1.15,
                early_stopping=True,
            )
            translation = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
            
        elif model_type == 'mt5':
            prompt = build_prompt(text, examples, model_type='mt5')
            inputs = tokenizer(prompt, return_tensors="pt", max_length=1024, truncation=True).to(model.device)
            
            outputs = model.generate(
                **inputs,
                max_new_tokens=Config.MT5_MAX_TOKENS,
                num_beams=Config.MT5_BEAMS,
                length_penalty=Config.LENGTH_PENALTY,
                repetition_penalty=Config.REPETITION_PENALTY,
                no_repeat_ngram_size=Config.NO_REPEAT_NGRAM,
                early_stopping=Config.EARLY_STOPPING,
            )
            translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
            
        else:  # ByT5
            prompt = build_prompt(text, examples, model_type='byt5')
            inputs = tokenizer(prompt, return_tensors="pt", max_length=1536, truncation=True).to(model.device)
            input_length = inputs['input_ids'].shape[1]
            max_new = min(Config.MAX_NEW_TOKENS, 2048 - input_length - 50)
            
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new,
                min_length=Config.MIN_LENGTH,
                num_beams=Config.NUM_BEAMS,
                length_penalty=Config.LENGTH_PENALTY,
                early_stopping=Config.EARLY_STOPPING,
                repetition_penalty=Config.REPETITION_PENALTY,
                no_repeat_ngram_size=Config.NO_REPEAT_NGRAM,
            )
            translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        return clean_translation(translation)
        
    except Exception as e:
        print(f"⚠️  {model_dict['name']} generation error: {str(e)[:50]}")
        return ""

def clean_translation(text):
    if not text:
        return ""
    
    # Remove artifacts
    text = re.sub(r'^.*?English:\s*', '', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'^.*?=\s*', '', text)
    text = re.sub(r'^\d+\.\s*', '', text)
    text = re.sub(r'^[Aa]kkadian[:\s]*', '', text)
    text = re.sub(r'^[Ee]nglish[:\s]*', '', text)
    
    # Stop at markers
    for marker in ["Akkadian:", "English:", "\n\n", "translate"]:
        if marker in text:
            text = text.split(marker)[0]
    
    # Clean spacing
    text = re.sub(r'\s+([,.!?;:])', r'\1', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'([,.!?])\1+', r'\1', text)
    
    # Remove immediate word repetitions
    words = text.split()
    cleaned_words = []
    for i, word in enumerate(words):
        if i == 0 or word.lower() != words[i-1].lower():
            cleaned_words.append(word)
    text = ' '.join(cleaned_words)
    
    return text.strip()

def post_process(text):
    if not text or len(text) < 3:
        return ""
    
    text = text.strip()
    
    # Capitalize
    if text and text[0].islower():
        text = text[0].upper() + text[1:]
    
    # Add period
    if text and len(text.split()) >= 5 and text[-1] not in '.!?':
        text += '.'
    
    # Remove incomplete trailing fragments
    sentences = re.split(r'[.!?]', text)
    if len(sentences) > 1:
        last = sentences[-1].strip()
        if last and len(last.split()) < 3:
            text = '.'.join(sentences[:-1]) + '.'
    
    return text.strip()

# ============================================================================
# ENSEMBLE
# ============================================================================

def calculate_quality_score(text):
    if not text or len(text) < 5:
        return 0.0
    
    words = text.split()
    word_count = len(words)
    
    if word_count < 10:
        return 0.4
    if word_count > 150:
        return 0.6
    
    score = 1.0
    score += 0.2 if text[-1] in '.!?' else 0
    score += 0.15 if text[0].isupper() else 0
    score += min(word_count / 25, 0.4)
    
    if any(x in text.lower() for x in ['akkadian:', 'english:', 'translate', '###']):
        score *= 0.5
    
    sentence_count = len([s for s in re.split(r'[.!?]', text) if s.strip()])
    score += min(sentence_count * 0.1, 0.3)
    
    return max(0.1, min(score, 2.5))

def consensus_ensemble(translations):
    if not translations:
        return ""
    
    for t in translations:
        quality = calculate_quality_score(t['text'])
        t['quality'] = quality
        t['combined_score'] = t['weight'] * quality * (1.0 / (t['priority'] + 1))
    
    translations.sort(key=lambda x: x['combined_score'], reverse=True)
    
    if len(translations) == 1:
        return translations[0]['text']
    
    best = translations[0]
    second_best = translations[1]
    
    if best['combined_score'] > second_best['combined_score'] * 1.3:
        return best['text']
    
    quality_trans = [t for t in translations[:3] if t['quality'] > 0.8]
    if quality_trans:
        return max(quality_trans, key=lambda x: len(x['text'].split()))['text']
    
    return best['text']

def ensemble_translate(models, test_src, examples):
    translations = []
    
    for model_dict in models:
        try:
            trans = translate_with_model(model_dict, test_src, examples)
            trans = post_process(trans)
            
            if trans and len(trans.split()) >= Config.MIN_TRANSLATION_WORDS:
                translations.append({
                    'text': trans,
                    'weight': model_dict['weight'],
                    'priority': model_dict['priority'],
                    'name': model_dict['name']
                })
        except Exception as e:
            print(f"⚠️  {model_dict['name']} error: {str(e)[:50]}")
            continue
    
    if not translations:
        return "Translation unavailable."
    
    return consensus_ensemble(translations)

# ============================================================================
# MAIN PIPELINE
# ============================================================================

def main():
    print("\n" + "="*70)
    print("ENHANCED AKKADIAN-TO-ENGLISH TRANSLATION SYSTEM")
    print("="*70)
    
    train_df, test_df = load_data()
    train_features = precompute_features(train_df)
    models = load_all_models()
    
    print(f"\n🔹 Configuration:")
    print(f"   Models: {len(models)}")
    print(f"   Training examples: {len(train_df)}")
    print(f"   Test samples: {len(test_df)}")
    print(f"   Ensemble: {Config.ENSEMBLE_METHOD}")
    
    predictions = []
    
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Translating"):
        try:
            examples = retrieve_examples(
                train_df, 
                train_features, 
                row["src_norm"], 
                k=Config.NUM_EXAMPLES
            )
            
            translation = ensemble_translate(models, row["src"], examples)
            
            if not translation or len(translation.split()) < 3:
                translation = "The translation is not available."
            
            predictions.append(translation)
            
        except Exception as e:
            print(f"\n⚠️  Error on sample {idx}: {str(e)}")
            predictions.append("Translation error occurred.")
        
        if (idx + 1) % Config.MEMORY_CLEANUP_EVERY == 0:
            torch.cuda.empty_cache()
            gc.collect()
    
    submission = pd.DataFrame({
        "id": test_df["id"], 
        "translation": predictions
    })
    submission.to_csv("submission.csv", index=False)
    
    print("\n" + "="*70)
    print("SAMPLE TRANSLATIONS")
    print("="*70)
    
    for i in range(min(5, len(test_df))):
        print(f"\n[{i}] {predictions[i]}")
    
    avg_length = np.mean([len(p.split()) for p in predictions if p])
    median_length = np.median([len(p.split()) for p in predictions if p])
    
    print(f"\n📊 Statistics:")
    print(f"   Average length: {avg_length:.1f} words")
    print(f"   Median length: {median_length:.1f} words")
    print(f"   Total samples: {len(predictions)}")
    
    print(f"\n✅ Submission saved to submission.csv")
    print("="*70)

if __name__ == "__main__":
    main()

E0000 00:00:1768107130.198330      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768107130.247347      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768107130.665279      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768107130.665312      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768107130.665316      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768107130.665318      24 computation_placer.cc:177] computation placer already registered. Please check linka


ENHANCED AKKADIAN-TO-ENGLISH TRANSLATION SYSTEM
🔹 Loading data...
✓ Train: 1385 | Test: 4
🔹 Pre-computing features...


`torch_dtype` is deprecated! Use `dtype` instead!


✓ Features ready

🔹 Loading models...
   Loading ByT5-Combined...
   ✓ ByT5-Combined loaded
   Loading ByT5-BigData2...
   ✓ ByT5-BigData2 loaded
   Loading ByT5-Akkadian...
   ✓ ByT5-Akkadian loaded
   Loading mT5-Small...
   ✓ mT5-Small loaded

✓ 4 model(s) ready:
   • ByT5-Combined (weight: 1.5, type: byt5)
   • ByT5-BigData2 (weight: 1.4, type: byt5)
   • ByT5-Akkadian (weight: 1.3, type: byt5)
   • mT5-Small (weight: 1.2, type: mt5)

🔹 Configuration:
   Models: 4
   Training examples: 1385
   Test samples: 4
   Ensemble: consensus_weighted


Translating:   0%|          | 0/4 [00:00<?, ?it/s]


SAMPLE TRANSLATIONS

[0] Thus Kuliya, the messenger of Kanesh, and Durhumit, say to Asānum, our missive, in particular, when you heard about having left a caravan against Kūn-išku: Aššur-mēssī is seized for two ordinary statements. He sha.

[1] Of the 8 minas of refined silver belonging to Elalī, son oF Ištar-pilah, in hahhum, Ennam-Aššur wrote about and his tablet was: In accordance with what creditor, A.

[2] When Nuhšātum and Ennam-Aššur wrote to Kanesh, as for the proceedings of Amur-Assyria, sent you a lawsuit about which I have not brought from him, within two months,. we shall return.

[3] Dan-Aššur seized us against Enna-Suen son of Iddin-abum and he gave him to hanu, they said: 'You shall return a man'. If yourselves do not raise claim, bring it for me as witnesses. Also, what owes, is in my presence o.

📊 Statistics:
   Average length: 33.5 words
   Median length: 32.5 words
   Total samples: 4

✅ Submission saved to submission.csv
